# Shor-style integer factoring with Qiskit 2.5

This notebook is a **learning-first implementation** of Shor-style factoring in Google Colab using modern Qiskit. It is designed to do two things at once: teach the flow of the algorithm at a high level, and provide a notebook that can actually factor small composite integers by combining a quantum order-finding subroutine with classical post-processing.

A practical warning is important up front: a fully generic, scalable implementation of Shor's algorithm needs efficient modular exponentiation circuits, and those become large very quickly. IBM's current Shor tutorial focuses on factoring 15 and explains that larger demonstrations such as 21 and 35 on present-day hardware are typically **compiled or specially tailored** rather than fully generic textbook implementations. Qiskit SDK v2.5 is current and keeps compatibility expectations stable for most users, while IBM's documentation still presents the main Shor tutorial around the 15-example and notes that Qiskit SDK v2.0 or later is sufficient for that workflow. [web:1][page:2]

So the notebook below takes a realistic approach:
- It supports factoring arbitrary small odd composite inputs entered by you.
- It uses a **generic permutation-based modular multiplication gate** so the same code path works for many small values of \(N\).
- It performs the final factor extraction classically from the measured order candidates.
- It is intended for simulator use in Colab, because that is the most reliable setting for a notebook that should actually work repeatedly.

For numbers like 15 and 21, this approach is often usable on a simulator. For 35, success is much less guaranteed because circuit size grows rapidly and generic synthesized unitaries become expensive. IBM's tutorial also highlights that directly synthesizing modular multiplication from a permutation matrix can produce significantly deeper circuits than hand-optimized constructions. [page:2]

## What this notebook will do

In this section we set expectations for the workflow. Shor's algorithm reduces factoring to an **order-finding** problem, then uses that order to recover nontrivial factors with the greatest common divisor computation.

Expected outcome:
- You will enter an integer such as 15 or 21.
- The notebook will try several random coprime bases \(a\).
- For each base, it will build an order-finding circuit, run it on Aer, convert measured phases to rational approximations, and test whether the inferred order gives valid factors.
- When successful, it will print factors such as \((3,5)\) for 15 or \((3,7)\) for 21.

In [ ]:
# If you are in Google Colab, run this cell once.
# Qiskit SDK v2.5 is the current major minor release according to IBM's release summary.
# Aer is used for simulation.

%pip -q install "qiskit>=2.5" "qiskit-aer>=0.17" pylatexenc matplotlib pandas

## Imports and environment check

The next code cell imports the libraries used throughout the notebook. We use:
- `QuantumCircuit` and `QFT` from Qiskit to build the order-finding circuit,
- `UnitaryGate` to create modular multiplication as a permutation unitary,
- `AerSimulator` to execute the circuit in Colab,
- standard Python modules such as `math`, `fractions`, and `random` for the classical part.

Expected output:
- The installed Qiskit version is printed.
- No quantum result appears yet; this is only setup.

In [ ]:
import math
import random
from fractions import Fraction

import numpy as np
import pandas as pd
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit.library import QFT
from qiskit.circuit.library import UnitaryGate
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

print("Qiskit version:", qiskit.__version__)

## High-level idea

Before writing helper functions, it helps to keep the logic straight.

Shor's algorithm uses the fact that if we choose a number \(a\) that is coprime to \(N\), then the sequence \(a^x mod N\) is periodic. The smallest positive period \(r\) such that \(a^r \equiv 1 \pmod N\) is called the **order** of \(a\) modulo \(N\). IBM's tutorial explains that phase estimation can recover information about \(k/r\), and from that we can often infer \(r\). Once we have an even order \(r\) with \(a^{r/2} 
ot\equiv -1 \pmod N\), the factors can be extracted using gcd computations based on \(a^{r/2} \pm 1\). [page:2]

Expected output:
- This is a markdown explanation only.
- The next cells will implement the classical and quantum utilities that match this logic.

## Classical helper functions

The next cell builds the classical utilities used by the rest of the notebook.

What each function does:
- `is_prime_trial`: small trial-division primality test so we can reject prime inputs.
- `prime_power_factor`: catches easy cases such as \(9 = 3^2\) or \(27 = 3^3\).
- `pick_random_coprime`: chooses a random base \(a\) with \(\gcd(a,N)=1\).
- `continued_fraction_order_candidates`: converts measured phases into denominator guesses for the order.
- `factors_from_order`: tests whether a guessed order really yields nontrivial factors.

Expected output:
- No visible output by default, because these are definitions.

In [ ]:
def is_prime_trial(n: int) -> bool:
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    d = 3
    while d * d <= n:
        if n % d == 0:
            return False
        d += 2
    return True


def prime_power_factor(n: int):
    for p in range(2, int(round(math.sqrt(n))) + 2):
        if is_prime_trial(p):
            x = p
            while x < n:
                x *= p
            y = p
            while y <= n:
                if y == n:
                    return p, n // p
                y *= p
    return None


def pick_random_coprime(n: int, rng=None) -> int:
    rng = rng or random
    while True:
        a = rng.randrange(2, n)
        if math.gcd(a, n) == 1:
            return a


def continued_fraction_order_candidates(phases, n, max_denominator=None):
    max_denominator = max_denominator or n
    candidates = []
    for phase in phases:
        frac = Fraction(phase).limit_denominator(max_denominator)
        r = frac.denominator
        if r not in candidates:
            candidates.append(r)
    return candidates


def factors_from_order(a: int, n: int, r: int):
    if r <= 0 or r % 2 == 1:
        return None
    x = pow(a, r // 2, n)
    if x == n - 1 or x == 1:
        return None
    p = math.gcd(x - 1, n)
    q = math.gcd(x + 1, n)
    if 1 < p < n and 1 < q < n and p * q == n:
        return tuple(sorted((p, q)))
    if 1 < p < n:
        return tuple(sorted((p, n // p)))
    if 1 < q < n:
        return tuple(sorted((q, n // q)))
    return None

## Building modular multiplication as a unitary

This is the key generic step. IBM's Shor tutorial notes that for small hand-crafted examples like \(N=15\), one can manually build modular multiplication gates using simple swap patterns, and that these are much shallower than synthesizing the whole permutation matrix automatically. It also shows that the generic permutation-matrix approach is valid, but can become significantly deeper after transpilation. [page:2]

Here we intentionally choose the generic route because it lets the notebook accept many small user inputs through a single implementation. The cost is circuit depth, so this notebook is best thought of as an educational simulator notebook rather than an optimized research-grade factoring engine.

Expected output:
- The code defines a gate constructor only.

In [ ]:
def num_target_qubits(n: int) -> int:
    return math.ceil(math.log2(n))


def modular_multiplication_unitary(b: int, n: int) -> np.ndarray:
    m = num_target_qubits(n)
    dim = 2 ** m
    U = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        if x < n:
            y = (b * x) % n
            U[y, x] = 1.0
        else:
            U[x, x] = 1.0
    return U


def modular_multiplication_gate(b: int, n: int) -> UnitaryGate:
    if math.gcd(b, n) != 1:
        raise ValueError(f"b={b} is not coprime with n={n}, so multiplication is not unitary on Z_n^*.")
    U = modular_multiplication_unitary(b, n)
    gate = UnitaryGate(U, label=f"M_{b} mod {n}")
    return gate

## Order-finding circuit constructor

Now we create the phase-estimation-style circuit. IBM's tutorial describes the same conceptual structure: a control register prepared with Hadamards, controlled powers of modular multiplication, an inverse QFT on the control register, and final measurement of the control qubits. The measured bitstrings approximate phases related to \(k/r\). [page:2]

The one technical choice we make here is the number of control qubits. A common educational default is to use about \(2m\) control qubits when the target register uses \(m\) qubits, which gives enough precision for many small examples.

Expected output:
- The cell defines the circuit-building function.

In [ ]:
def build_order_finding_circuit(a: int, n: int, num_control=None):
    m = num_target_qubits(n)
    if num_control is None:
        num_control = 2 * m

    control = QuantumRegister(num_control, name="c")
    target = QuantumRegister(m, name="t")
    classical = ClassicalRegister(num_control, name="meas")
    qc = QuantumCircuit(control, target, classical)

    # Prepare |1> in the target register.
    qc.x(target[0])

    # Apply Hadamards and controlled modular multiplications by a^(2^k) mod N.
    current = a % n
    for k in range(num_control):
        qc.h(control[k])
        gate = modular_multiplication_gate(current, n).control(1)
        qc.append(gate, [control[k], *target])
        current = pow(current, 2, n)

    qc.append(QFT(num_control, inverse=True, do_swaps=True), control)
    qc.measure(control, classical)
    return qc

## Running the quantum subroutine

This section adds the execution helper. We transpile for `AerSimulator`, run the circuit for a chosen number of shots, and convert the measurement counts into phase values.

Expected output:
- The cell defines helper functions.
- No circuit is executed yet.

In [ ]:
def bitstring_to_phase(bitstring: str) -> float:
    return int(bitstring, 2) / (2 ** len(bitstring))


def run_order_finding(a: int, n: int, num_control=None, shots=2048, seed=7):
    simulator = AerSimulator(seed_simulator=seed)
    qc = build_order_finding_circuit(a, n, num_control=num_control)
    tqc = transpile(qc, simulator, optimization_level=1)
    result = simulator.run(tqc, shots=shots).result()
    counts = result.get_counts()

    phases = []
    rows = []
    for bitstring, count in sorted(counts.items(), key=lambda kv: kv[1], reverse=True):
        phase = bitstring_to_phase(bitstring)
        phases.append(phase)
        rows.append({
            "bitstring": bitstring,
            "count": count,
            "phase": phase,
            "fraction_guess": str(Fraction(phase).limit_denominator(n))
        })

    df = pd.DataFrame(rows)
    return qc, tqc, counts, phases, df

## Full factoring wrapper

The next cell combines the easy classical checks with the quantum order-finding loop.

Flow of the function:
1. Reject invalid inputs such as even numbers, numbers below 3, or primes.
2. Catch lucky classical shortcuts, for example when a chosen base already shares a gcd with \(N\).
3. Otherwise run the quantum order-finding circuit for several random coprime bases.
4. Convert dominant measured phases into candidate orders.
5. Test those candidate orders and return the first nontrivial factors found.

Expected output:
- The factoring function becomes available for direct use.

In [ ]:
def shor_factor(n: int, max_attempts=8, shots=2048, top_k_phases=12, seed=7, verbose=True):
    if n < 3:
        raise ValueError("Input must be an integer greater than 2.")
    if n % 2 == 0:
        return {
            "success": True,
            "method": "classical_even_check",
            "factors": (2, n // 2)
        }
    if is_prime_trial(n):
        return {
            "success": False,
            "method": "classical_prime_check",
            "message": f"{n} is prime, so there are no nontrivial factors."
        }

    pp = prime_power_factor(n)
    if pp is not None:
        return {
            "success": True,
            "method": "classical_prime_power_check",
            "factors": tuple(sorted(pp))
        }

    rng = random.Random(seed)

    for attempt in range(1, max_attempts + 1):
        a = pick_random_coprime(n, rng=rng)
        g = math.gcd(a, n)
        if g > 1:
            return {
                "success": True,
                "method": "lucky_classical_gcd",
                "factors": tuple(sorted((g, n // g))),
                "attempt": attempt,
                "a": a
            }

        qc, tqc, counts, phases, df = run_order_finding(a, n, shots=shots, seed=seed + attempt)
        dominant_phases = df.head(top_k_phases)["phase"].tolist()
        candidate_orders = continued_fraction_order_candidates(dominant_phases, n)

        if verbose:
            print(f"Attempt {attempt}: a = {a}")
            print("Candidate orders from dominant phases:", candidate_orders)

        for r in candidate_orders:
            factors = factors_from_order(a, n, r)
            if factors is not None:
                return {
                    "success": True,
                    "method": "quantum_order_finding",
                    "factors": factors,
                    "attempt": attempt,
                    "a": a,
                    "r": r,
                    "raw_counts": counts,
                    "results_table": df,
                    "circuit": qc,
                    "transpiled_circuit": tqc,
                }

    return {
        "success": False,
        "method": "quantum_order_finding",
        "message": "No nontrivial factors found within the attempt budget. Try increasing shots or attempts.",
    }

## User input cell

This is the cell you will edit most often. Change `N_TO_FACTOR` to the integer you want to test.

Recommended starting values:
- 15, because IBM's main tutorial uses it and it is the standard demonstration example. [page:2]
- 21, because it is a natural next challenge and also appears in the discussion section of IBM's tutorial as a demonstrated compiled case. [page:2]
- 35, but treat it as an experiment rather than a promise, because the generic synthesized-unitary approach becomes much heavier.

Expected output:
- Only the chosen value is stored.

In [ ]:
N_TO_FACTOR = 15
MAX_ATTEMPTS = 8
SHOTS = 2048
TOP_K_PHASES = 12
SEED = 7

## Run the factoring experiment

This is the main execution cell. It calls the wrapper and prints the result dictionary.

Expected output:
- For easy cases like 15, you should usually see success and a pair of nontrivial factors.
- For some runs or larger inputs, the attempt budget may be exhausted. That does not necessarily mean the theory failed; it often means the estimated order from finite-shot data was not good enough this time.

In [ ]:
result = shor_factor(
    N_TO_FACTOR,
    max_attempts=MAX_ATTEMPTS,
    shots=SHOTS,
    top_k_phases=TOP_K_PHASES,
    seed=SEED,
    verbose=True,
)

print("
Final result:")
for key, value in result.items():
    if key not in {"raw_counts", "results_table", "circuit", "transpiled_circuit"}:
        print(f"{key}: {value}")

## Inspect the measurement distribution

If the factoring run succeeded through the quantum path, the next cell plots the measurement histogram. This helps you connect the abstract phase-estimation story to actual sampled outputs.

Expected output:
- A histogram of measured control-register bitstrings.
- The most frequent strings usually correspond to phases close to rational values with denominator related to the order.

In [ ]:
if result.get("success") and result.get("method") == "quantum_order_finding":
    display(result["results_table"].head(15))
    display(plot_histogram(result["raw_counts"], figsize=(12, 4)))
else:
    print("No quantum histogram available for this run.")

## View the generated circuit

This cell shows the logical circuit before and after transpilation. The pre-transpiled view is easier to understand conceptually, while the transpiled view reveals the implementation cost. IBM's documentation emphasizes that generic synthesized modular multiplication can become much deeper than tailored constructions, and this comparison is one of the main learning moments in the notebook. [page:2]

Expected output:
- A readable logical circuit drawing.
- A transpiled circuit depth report if quantum execution occurred.

In [ ]:
if result.get("success") and result.get("method") == "quantum_order_finding":
    qc = result["circuit"]
    tqc = result["transpiled_circuit"]
    print("Logical circuit depth:", qc.depth())
    print("Transpiled circuit depth:", tqc.depth())
    print("Logical qubits:", qc.num_qubits)
    print("Transpiled operation counts:", tqc.count_ops())
    display(qc.draw("mpl", fold=120))
else:
    print("No quantum circuit object available for display.")

## Notes on 15, 21, and 35

This final discussion cell explains what to expect as you push the notebook.

- **15** is the safest demonstration case. IBM's current official tutorial is built around factoring 15 and uses specially designed modular multiplication gates for that example. [page:2]
- **21** is a meaningful next target, but practical demonstrations usually rely on compiled or problem-tailored circuits rather than a broad fully generic implementation. IBM's tutorial explicitly challenges readers to try 21 after learning the 15 workflow. [page:2]
- **35** is historically famous in experimental demonstrations, but reaching it with a generic permutation-synthesis notebook is difficult. The issue is not the mathematics of Shor's algorithm; it is the rapidly increasing circuit cost of modular multiplication and its controlled powers on current tools and devices. IBM's tutorial also notes that present-day hardware is still far from the scale needed for cryptographically large integers, and even small demonstrations often need optimization tricks or compiled constructions. [page:2]

If you want to improve this notebook later, the best next step is not to change the classical post-processing. The best next step is to replace the generic `UnitaryGate` modular multiplication with hand-optimized arithmetic circuits or compiled constructions for specific families of \(N\).